In [2]:
import re
import unicodedata
from typing import Optional, List, Tuple, Set

# -----------------------
# Configuration & Constants
# -----------------------

ARABIC_NORMALIZATION_MAP = {
    # Alef variants
    "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ٲ": "ا", "ٳ": "ا",
    # Ya variants
    "ى": "ي", "ئ": "ي", "ۍ": "ي", "ێ": "ي",
    # Waw variants
    "ؤ": "و", "ۆ": "و",
    # Ta Marbuta
    "ة": "ه", "ۃ": "ه",
    # Ha variants
    "ھ": "ه",
}

ARABIC_DIACRITICS_PATTERN = re.compile(
    r"[\u064B-\u065F\u0670\u06D6-\u06ED\u08D4-\u08E1\u08D3-\u08FF\uFE70-\uFEFF]"
)

TITLES = {
    # English titles
    "mr", "mr.", "mrs", "mrs.", "ms", "ms.", "miss", "mister", "mistress",
    "dr", "dr.", "doctor", "prof", "prof.", "professor",
    "eng", "eng.", "engineer", "sir", "lady", "lord", "dame",
    "capt", "capt.", "captain", "col", "col.", "colonel", "gen", "gen.", "general",
    "lt", "lt.", "lieutenant", "maj", "maj.", "major", "sgt", "sergeant",
    "rev", "rev.", "reverend", "fr", "fr.", "father", "sr", "sr.", "sister",
    "hon", "hon.", "honorable", "esq", "esq.", "esquire",
    # Arabic titles
    "باشا", "بشا", "بيه", "بك", "أفندي", "افندي",
    "دكتور", "د", "د.", "دكتوره",
    "مهندس", "مهندسه", "م", "م.", "مهندسة",
    "أستاذ", "استاذ", "أ", "أ.", "ا", "ا.", "استاذه", "أستاذة",
    "شيخ", "الشيخ", "شيخة",
    "سيد", "سيدة", "آنسة", "انسة", "السيد", "السيدة",
    "حاج", "حاجة", "الحاج", "الحاجة", "حاجه",
    "قائد", "رائد", "عميد", "لواء", "فريق",
}

NOISE_WORDS = {
    # English noise
    "co", "co.", "company", "corp", "corp.", "corporation", "inc", "inc.", "incorporated",
    "group", "sons", "son", "and", "&", "the", "of", "for",
    "ltd", "ltd.", "limited", "plc", "llc", "llc.", "gmbh",
    "associates", "associate", "brothers", "brother", "bros", "bros.",
    "establishment", "est", "est.", "enterprise", "enterprises",
    # Arabic noise
    "بن", "ابن", "ابو", "أبو", "أبي", "ابي",
    "آل", "ال", "ال.", "و", "و.", "من",
    "شركة", "شركه", "شركات", "مجموعة", "مجموعه", "مجموعات",
    "واولاده", "وأولاده", "وشركاه", "وشركائه", "وأولاد",
    "مؤسسة", "مؤسسه", "مكتب", "بيت",
    # Common geographic/nationality markers
    "المصري", "المصريه", "المصرية", "مصري", "مصرية", "مصريه",
    "السعودي", "السعودية", "سعودي", "سعودية", "سعوديه",
    "اللبناني", "اللبنانية", "لبناني", "لبنانية", "لبنانيه",
    "الاردني", "الأردني", "اردني", "أردني",
    "السوري", "سوري", "سورية", "سوريه",
    "العراقي", "عراقي", "عراقية",
    "الكويتي", "كويتي", "كويتية",
    "الاماراتي", "الإماراتي", "اماراتي", "إماراتي",
}

# Extended compound names
COMPOUND_NAMES = [
    # Abd/Abdul variants - Rahman
    ("abdel", "rahman", "abdelrahman"), ("abdul", "rahman", "abdulrahman"),
    ("abd", "rahman", "abdrahman"), ("abdal", "rahman", "abdalrahman"),
    ("عبد", "الرحمن", "عبدالرحمن"), ("عبد", "رحمن", "عبدالرحمن"),
    # Abd/Abdul variants - Aziz
    ("abdel", "aziz", "abdelaziz"), ("abdul", "aziz", "abdulaziz"),
    ("abd", "aziz", "abdaziz"), ("عبد", "العزيز", "عبدالعزيز"),
    ("عبد", "عزيز", "عبدالعزيز"),
    # Abd/Abdul variants - Other
    ("abdel", "kader", "abdelkader"), ("abdul", "kader", "abdulkader"),
    ("abdel", "hamid", "abdelhamid"), ("abdul", "hamid", "abdulhamid"),
    ("abdel", "latif", "abdellatif"), ("abdul", "latif", "abdullatif"),
    ("abdel", "moneim", "abdelmoneim"), ("abdul", "moneim", "abdulmoneim"),
    ("abdel", "salam", "abdelsalam"), ("abdul", "salam", "abdulsalam"),
    ("abdel", "fattah", "abdelfattah"), ("abdul", "fattah", "abdulfattah"),
    ("abdel", "wahab", "abdelwahab"), ("abdul", "wahab", "abdulwahab"),
    ("abdel", "malik", "abdelmalik"), ("abdul", "malik", "abdulmalik"),
    ("abdel", "nasser", "abdelnasser"), ("abdul", "nasser", "abdulnasser"),
    ("abdel", "halim", "abdelhalim"), ("abdul", "halim", "abdulhalim"),
    ("abdel", "karim", "abdelkarim"), ("abdul", "karim", "abdulkarim"),
    ("abdel", "majid", "abdelmajid"), ("abdul", "majid", "abdulmajid"),
    # Abu compounds
    ("abu", "baker", "abubaker"), ("abu", "bakr", "abubakr"),
    ("abu", "el", "abuel"), ("أبو", "بكر", "أبوبكر"),
    # Mohamed/Ali compounds
    ("mohamed", "ali", "mohamedali"), ("mohammed", "ali", "mohammedali"),
    ("محمد", "علي", "محمدعلي"),
    # Arabic Abd compounds
    ("عبد", "الله", "عبدالله"), ("عبد", "الكريم", "عبدالكريم"),
    ("عبد", "الحميد", "عبدالحميد"), ("عبد", "المجيد", "عبدالمجيد"),
    ("عبد", "الناصر", "عبدالناصر"), ("عبد", "الحليم", "عبدالحليم"),
]

# Enhanced keyboard error mapping (Arabic-English keyboard mix)
KEYBOARD_ERROR_MAP = {
    "s": "س", "h": "ه", "g": "ج", "d": "د", "f": "ف",
    "k": "ك", "l": "ل", "m": "م", "n": "ن", "t": "ت",
    "y": "ي", "a": "ا", "b": "ب", "e": "ع", "r": "ر",
    "z": "ز", "c": "ش", "x": "ض", "w": "ص", "q": "ق",
}

# Common name variations to standardize
NAME_VARIATIONS = {
    # Mohamed variants
    "mohammed": "mohamed", "mohamad": "mohamed", "muhammad": "mohamed",
    "mohammad": "mohamed", "muhamed": "mohamed", "muhammed": "mohamed",
    "muhamad": "mohamed", "mouhammad": "mohamed", "mohammod": "mohamed",
    "محمد": "mohamed", "محمود": "mahmud",
    # Ahmed variants
    "ahmad": "ahmed", "ahmmed": "ahmed", "ahmet": "ahmed",
    "ahmead": "ahmed", "احمد": "ahmed",
    # Other common variations
    "mahmoud": "mahmud", "mahmood": "mahmud", "mahmod": "mahmud",
    "hussein": "husein", "hossein": "husein", "husain": "husein", "hussain": "husein",
    "hasan": "hassan", "haasan": "hassan", "حسن": "hassan", "حسان": "hassan",
    "osama": "usama", "ousama": "usama", "اسامة": "usama", "أسامة": "usama",
    "uthman": "othman", "osman": "othman", "عثمان": "othman",
    "ibrahim": "ibrahim", "ibraheem": "ibrahim", "ابراهيم": "ibrahim", "إبراهيم": "ibrahim",
    "youssef": "yousef", "yusuf": "yousef", "youssif": "yousef", "يوسف": "yousef",
}

# Common data entry mistakes
DATA_ENTRY_FIXES = {
    # Space errors
    r"\s+": " ",  # Multiple spaces
    r"^\s+|\s+$": "",  # Leading/trailing spaces
    # Punctuation duplication
    r"\.{2,}": "",  # Multiple periods
    r",{2,}": "",  # Multiple commas
    r"-{2,}": " ",  # Multiple hyphens
    r"_{2,}": " ",  # Multiple underscores
    # Common typos
    r"(\w)\1{3,}": r"\1",  # 4+ repeated characters
}

# Characters to remove entirely
REMOVE_CHARS = {
    # Special symbols
    '~', '`', '!', '@', '#', '$', '%', '^', '*', '(', ')', 
    '+', '=', '[', ']', '{', '}', '|', '\\', ':', ';', '"', 
    "'", '<', '>', '?', '/', '،', '؛', '؟',
    # Zero-width and invisible characters
    '\u200b', '\u200c', '\u200d', '\u200e', '\u200f',
    '\ufeff', '\u00a0', '\u202a', '\u202b', '\u202c', '\u202d', '\u202e',
}

# -----------------------
# Core Normalization Functions
# -----------------------

def normalize_arabic(text: str) -> str:
    """Normalize Arabic character variants to canonical forms."""
    if not text:
        return ""
    for original, normalized in ARABIC_NORMALIZATION_MAP.items():
        text = text.replace(original, normalized)
    return text

def remove_diacritics(text: str) -> str:
    """Remove all Arabic diacritical marks (tashkeel)."""
    return re.sub(ARABIC_DIACRITICS_PATTERN, "", text)

def normalize_english(text: str) -> str:
    """Normalize English text variations."""
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1", text)  # Remove excessive repetition
    tokens = text.split()
    normalized_tokens = [NAME_VARIATIONS.get(t, t) for t in tokens]
    return " ".join(normalized_tokens)

def remove_titles(text: str) -> str:
    """Remove honorific titles from names."""
    tokens = text.split()
    return " ".join(t for t in tokens if t.lower() not in TITLES and t not in TITLES)

def remove_noise_words(text: str) -> str:
    """Remove common noise words and business suffixes."""
    tokens = text.split()
    return " ".join(t for t in tokens if t.lower() not in NOISE_WORDS and t not in NOISE_WORDS)

def fix_keyboard_errors(text: str, aggressive: bool = False) -> str:
    """Fix common keyboard layout errors (Arabic-English confusion)."""
    if not aggressive:
        return text
    tokens = text.split()
    fixed = []
    for token in tokens:
        has_arabic = any('\u0600' <= c <= '\u06FF' for c in token)
        has_english = any('a' <= c.lower() <= 'z' for c in token)
        if has_arabic and has_english:
            for eng, ara in KEYBOARD_ERROR_MAP.items():
                token = token.replace(eng, ara)
        fixed.append(token)
    return " ".join(fixed)

def merge_compound_names(tokens: List[str]) -> List[str]:
    """Merge compound names (e.g., 'abdel rahman' → 'abdelrahman')."""
    i = 0
    merged = []
    while i < len(tokens):
        merged_flag = False
        for first, second, compound in COMPOUND_NAMES:
            if i + 1 < len(tokens) and tokens[i] == first and tokens[i + 1] == second:
                merged.append(compound)
                i += 2
                merged_flag = True
                break
        if not merged_flag:
            merged.append(tokens[i])
            i += 1
    return merged

def remove_short_tokens(tokens: List[str], min_length: int = 2) -> List[str]:
    """Remove very short tokens that are likely noise."""
    return [t for t in tokens if len(t) >= min_length]

def remove_numeric_tokens(tokens: List[str]) -> List[str]:
    """Remove tokens that are purely numeric."""
    return [t for t in tokens if not t.isdigit()]

def handle_null_like_values(text: str) -> str:
    """Handle NULL-like values and placeholders."""
    null_values = {"null", "none", "n/a", "na", "nil", "undefined", "unknown", "#n/a", "#null", "غير معروف", "غير_معروف", "لا يوجد"}
    if text.lower().strip() in null_values:
        return ""
    return text

def remove_unwanted_characters(text: str) -> str:
    """Remove special characters and symbols."""
    for char in REMOVE_CHARS:
        text = text.replace(char, ' ')
    return text

def fix_common_data_entry_errors(text: str) -> str:
    """Fix common data entry mistakes."""
    for pattern, replacement in DATA_ENTRY_FIXES.items():
        text = re.sub(pattern, replacement, text)
    return text

def normalize_separators(text: str) -> str:
    """Normalize different types of separators to spaces."""
    separators = ['-', '_', '.', '/', '\\', '|', '،', '؛']
    for sep in separators:
        text = text.replace(sep, ' ')
    return text

def handle_mixed_scripts(text: str) -> str:
    """Handle text with mixed Arabic/English scripts."""
    # Detect if text has both scripts
    has_arabic = bool(re.search(r'[\u0600-\u06FF]', text))
    has_english = bool(re.search(r'[a-zA-Z]', text))
    if has_arabic and has_english:
        # Try to separate clearly Arabic from English parts
        parts = re.split(r'(\s+)', text)
        return ' '.join(parts)
    return text

def remove_html_entities(text: str) -> str:
    """Remove HTML entities and tags."""
    text = re.sub(r'&[a-zA-Z]+;', '', text)  # &nbsp; &amp; etc
    text = re.sub(r'&#\d+;', '', text)  # &#160; etc
    text = re.sub(r'<[^>]+>', '', text)  # HTML tags
    return text

def handle_encoding_issues(text: str) -> str:
    """Handle common encoding issues."""
    try:
        # Fix mojibake and encoding issues
        if isinstance(text, bytes):
            text = text.decode('utf-8', errors='ignore')
        # Remove non-printable characters except spaces
        text = ''.join(char for char in text if char.isprintable() or char.isspace())
    except:
        pass
    return text

# -----------------------
# Main Preprocessing Pipeline
# -----------------------

def preprocess_name(text: str, *, remove_duplicates: bool = True, sort_tokens: bool = True, fix_keyboards: bool = False, min_token_length: int = 2, remove_numbers: bool = True, preserve_order: bool = False) -> str:
    """Main preprocessing pipeline for Arabic-English names with comprehensive edge case handling."""
    if not text or not isinstance(text, str):
        return ""
    
    # Handle NULL-like values first
    text = handle_null_like_values(text)
    if not text:
        return ""
    
    # Handle encoding issues
    text = handle_encoding_issues(text)
    
    # Remove HTML entities
    text = remove_html_entities(text)
    
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)
    
    # Fix common data entry errors
    text = fix_common_data_entry_errors(text)
    
    # Remove unwanted special characters
    text = remove_unwanted_characters(text)
    
    # Normalize separators
    text = normalize_separators(text)
    
    # Handle mixed scripts
    text = handle_mixed_scripts(text)
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    if not text:
        return ""
    
    # Arabic-specific normalization
    text = normalize_arabic(text)
    text = remove_diacritics(text)
    
    # English normalization
    text = normalize_english(text)
    
    # Optional keyboard error fixing
    if fix_keyboards:
        text = fix_keyboard_errors(text, aggressive=True)
    
    # Remove titles and noise words
    text = remove_titles(text)
    text = remove_noise_words(text)
    
    # Lowercase everything
    text = text.lower()
    
    # Tokenize
    tokens = text.split()
    
    if not tokens:
        return ""
    
    # Remove duplicates while preserving order
    if remove_duplicates:
        seen = set()
        unique_tokens = []
        for t in tokens:
            if t not in seen:
                seen.add(t)
                unique_tokens.append(t)
        tokens = unique_tokens
    
    # Merge compound names
    tokens = merge_compound_names(tokens)
    
    # Remove short tokens
    tokens = remove_short_tokens(tokens, min_length=min_token_length)
    
    # Remove numeric tokens
    if remove_numbers:
        tokens = remove_numeric_tokens(tokens)
    
    # Final validation - remove empty strings
    tokens = [t for t in tokens if t and t.strip()]
    
    if not tokens:
        return ""
    
    # Sort for canonical form (unless preserving order)
    if sort_tokens and not preserve_order:
        tokens = sorted(tokens)
    
    # Join and return as single line
    return " ".join(tokens)

def preprocess_name_variants(text: str) -> List[str]:
    """Generate multiple preprocessed variants of a name for fuzzy matching."""
    variants = [preprocess_name(text), preprocess_name(text, fix_keyboards=True), preprocess_name(text, sort_tokens=False), preprocess_name(text, remove_duplicates=False)]
    return list(dict.fromkeys(v for v in variants if v))

def compare_names(name1: str, name2: str) -> bool:
    """Compare two names after preprocessing."""
    return preprocess_name(name1) == preprocess_name(name2)

def get_name_similarity_score(name1: str, name2: str) -> float:
    """Calculate similarity score between two names (0.0 to 1.0)."""
    tokens1 = set(preprocess_name(name1).split())
    tokens2 = set(preprocess_name(name2).split())
    if not tokens1 or not tokens2:
        return 0.0
    intersection = tokens1 & tokens2
    union = tokens1 | tokens2
    return len(intersection) / len(union) if union else 0.0

In [3]:
import re
from typing import List, Set


class NameNormalizer:
    def __init__(self):
        # --- Language-specific rewrite rules ---
        self.rules = [
            # Arabic-origin names in English
            (r"kh", "h"),        # khaled → haled
            (r"gh", "g"),        # ghada → gada
            (r"th", "t"),        # thamer → tamer
            (r"ph", "f"),        # pharis → faris
            (r"aa", "a"),        # aamir → amir

            # Hebrew & Arabic shared variations
            (r"ch", "h"),        # chaim → haim
            (r"tz", "s"),        # tzion → sion
            (r"sh", "s"),        # shlomo → slomo

            # Farsi/Persian translit
            (r"gh", "q"),        # gheisar → qeisar
            (r"oo", "u"),        # soor → sur

            # Kurdish / Pashto
            (r"jw", "ju"),
            (r"zh", "j"),
            (r"sh", "s"),
            (r"rr", "r")
        ]

        # Vowel normalization (language independent)
        self.vowels = "aeiou"
        
    def normalize_basic(self, name: str) -> str:
        name = name.lower().strip()
        name = re.sub(r"[^a-z]", "", name)  # remove non-letters
        return name

    def remove_vowels(self, text: str) -> str:
        return re.sub(f"[{self.vowels}]", "", text)

    def apply_rewrite_rules(self, name: str) -> Set[str]:
        variants = {name}
        for pattern, repl in self.rules:
            for v in list(variants):
                new_v = re.sub(pattern, repl, v)
                if new_v != v:
                    variants.add(new_v)
        return variants

    # Sound-alike generator: soft/hard consonants
    def phonetic_variants(self, name: str) -> Set[str]:
        variants = {name}

        swap_groups = [
            ("k", "c", "q"),
            ("s", "z"),
            ("t", "d"),
            ("f", "ph"),
            ("h", "7"),   # Arabic dialect "ح" → h
        ]

        for group in swap_groups:
            for v in list(variants):
                for g in group:
                    if g in v:
                        for alt in group:
                            variants.add(v.replace(g, alt))
        return variants

    def expand_transliterations(self, name: str) -> Set[str]:
        variants = {name}
        # Arabic letter approximations
        arabic_groups = {
            "aa": ["a", "ah"],
            "u": ["o", "ou"],
            "i": ["ee", "y", "ei"]
        }
        for key, alts in arabic_groups.items():
            for v in list(variants):
                if key in v:
                    for a in alts:
                        variants.add(v.replace(key, a))
        return variants

    def generate_equivalents(self, name: str) -> Set[str]:
        n = self.normalize_basic(name)

        variants = set()
        variants.add(n)

        # apply all engines
        variants |= self.apply_rewrite_rules(n)
        variants |= self.phonetic_variants(n)
        variants |= self.expand_transliterations(n)

        # Vowel-deleted root (for matching systems like Soundex)
        variants.add(self.remove_vowels(n))

        # Clean final output
        clean_set = set(filter(None, variants))

        return clean_set


# Usage
norm = NameNormalizer()

print(norm.generate_equivalents("Mohammed"))


{'mhmmd', 'mo7ammed', 'mohammet', 'mo7ammet', 'mohammed'}


In [ ]:
import re
import unicodedata
from typing import Optional, List, Set

# =====================================================================
# Phonetic & Transliteration Normalizer (Replaces NAME_VARIATIONS)
# =====================================================================

class PhoneticNormalizer:
    """
    Normalizes English/transliterated name tokens into a canonical form
    using a series of phonetic and character-based rules.
    """
    def __init__(self):
        # Rules are applied in order. More specific rules should come first.
        self._PHONETIC_RULES = [
            # 1. Digraphs and common transliterations (most specific)
            (r'kh', 'h'),      # Khaled -> Haled
            (r'gh', 'g'),      # Ghada -> Gada
            (r'sh', 's'),      # Shadi -> Sadi
            (r'th', 't'),      # Thamer -> Tamer
            (r'ph', 'f'),      # Philip -> Filip
            (r'ch', 's'),      # Charbel -> Sarbel (can also be 'k', but 's' is common in names)
            (r'ou', 'u'),      # Mahmoud -> Mahmud
            (r'ei', 'i'),      # Leila -> Lila
            (r'ie', 'i'),      # Nadim -> Nadim
            
            # 2. Vowel Normalization and reduction
            (r'aa', 'a'),      # Aamer -> Amer
            (r'ee', 'i'),      # Jameel -> Jamil
            (r'oo', 'u'),      # Mahmood -> Mahmud
            (r'y', 'i'),       # Youssef -> Iussef
            (r'ea', 'i'),      # Jean -> Jin
            
            # 3. Consonant Normalization (map many-to-one)
            (r'q', 'k'),       # Tariq -> Tarik
            (r'c', 'k'),       # Carol -> Karol
            (r'z', 's'),       # Ziad -> Siad
            (r'w', 'u'),       # Walid -> Ualid (often a vowel sound)
            
            # 4. Remove silent/doubled letters
            (r'([a-z])\1+', r'\1'), # Tammam -> Tamam, Mohammed -> Mohamed
        ]

    def normalize_token(self, token: str) -> str:
        """Applies all phonetic rules to a single token."""
        if not token:
            return ""
            
        # Basic cleanup: lowercase and keep only letters
        token = re.sub(r'[^a-z]', '', token.lower())

        for pattern, replacement in self._PHONETIC_RULES:
            token = re.sub(pattern, replacement, token)
            
        return token

# Instantiate the normalizer once to be used globally
phonetic_normalizer = PhoneticNormalizer()


# =====================================================================
# CONFIGURATION & CONSTANTS (Streamlined)
# =====================================================================

ARABIC_NORMALIZATION_MAP = {
    "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ٲ": "ا", "ٳ": "ا",
    "ى": "ي", "ئ": "ي", "ۍ": "ي", "ێ": "ي",
    "ؤ": "و", "ۆ": "و",
    "ة": "ه", "ۃ": "ه", "ھ": "ه",
}
ARABIC_DIACRITICS_PATTERN = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED\u08D4-\u08E1\u08D3-\u08FF\uFE70-\uFEFF]")

TITLES = {
    "mr","mr.","mrs","mrs.","ms","ms.","miss","mister","dr","dr.","doctor", "prof","prof.",
    "professor","eng","eng.","engineer","sir","lady","lord","باشا","بيه","بك","افندي",
    "دكتور","د","د.","دكتوره","الدكتور","مهندس","م","م.","المهندس","أستاذ","استاذ",
    "أ","أ.","شيخ","الشيخ","سيد","سيدة","السيد","السيدة","حاج","الحاج",
}
NOISE_WORDS = {
    "co","co.","company","corp","corporation","group","sons","and","the","of","ltd",
    "limited","plc","llc","inc","بن","ابن","ابو","أبو","آل","ال","و","من","شركة",
    "مجموعة","مجموعه","واولاده","مؤسسة",
}
COMPOUND_NAMES = [
    ("abdel","rahman","abdelrahman"), ("abdul","rahman","abdulrahman"),
    ("abdel","aziz","abdelaziz"), ("abdul","aziz","abdulaziz"),
    ("عبد","الرحمن","عبدالرحمن"), ("عبد","العزيز","عبدالعزيز"),
    ("عبد","الله","عبدالله"), ("صلاح","الدين","صلاحالدين"),
    ("ابو","بكر","ابوبكر"),
]
NULL_LIKE_VALUES = {"null", "none", "n/a", "na", "nil", "undefined", "unknown", "--", "-", "غير معروف", "لا يوجد"}
DATA_ENTRY_FIXES = {r"\s+": " ", r"^\s+|\s+$": ""}
REMOVE_CHARS_PATTERN = re.compile(r"[~`!@#$%^&*()+=[]{}\\|:;\"'<>,?،؛؟«»\._\-/]")
INVISIBLE_CHARS_PATTERN = re.compile(r'[\u200b-\u200f\ufeff\u00a0]')

# =====================================================================
# CORE HELPER FUNCTIONS
# =====================================================================

def normalize_arabic(text: str) -> str:
    for old, new in ARABIC_NORMALIZATION_MAP.items():
        text = text.replace(old, new)
    return re.sub(ARABIC_DIACRITICS_PATTERN, "", text)

def merge_compound_names(tokens: List[str]) -> List[str]:
    # This function remains useful for merging tokens before phonetic analysis
    i = 0
    output_tokens = []
    first_parts = {c[0] for c in COMPOUND_NAMES}
    
    while i < len(tokens):
        if tokens[i] in first_parts and i + 1 < len(tokens):
            merged = False
            for first, second, compound in COMPOUND_NAMES:
                if tokens[i] == first and tokens[i+1] == second:
                    output_tokens.append(compound)
                    i += 2
                    merged = True
                    break
            if not merged:
                output_tokens.append(tokens[i])
                i += 1
        else:
            output_tokens.append(tokens[i])
            i += 1
    return output_tokens

# =====================================================================
# MAIN PIPELINE
# =====================================================================

def preprocess_name(
    name: Optional[str],
    *,
    remove_duplicates: bool = True,
    sort_tokens: bool = True,
    preserve_order: bool = False,
    min_token_length: int = 2,
) -> str:
    """
    Cleans, normalizes, and standardizes a name string using structural and phonetic rules.
    """
    # 1. Initial Sanity Checks and Cleaning
    if not name or not isinstance(name, str):
        return ""
    clean_name = name.lower().strip()
    if clean_name in NULL_LIKE_VALUES:
        return ""

    # 2. Character-level and Structural Cleaning
    try:
        clean_name = unicodedata.normalize("NFKC", clean_name)
    except Exception: pass
    clean_name = re.sub(r"<[^>]+>", " ", clean_name)
    clean_name = re.sub(INVISIBLE_CHARS_PATTERN, '', clean_name)
    clean_name = re.sub(REMOVE_CHARS_PATTERN, ' ', clean_name)
    clean_name = normalize_arabic(clean_name)
    clean_name = re.sub(r'\s+', ' ', clean_name).strip()

    # 3. Tokenization and Noise Removal
    tokens = clean_name.split()
    tokens = [t for t in tokens if t not in TITLES and t not in NOISE_WORDS and not t.isdigit()]
    
    # 4. Semantic and Phonetic Normalization
    # First, merge compound names like "abdel rahman"
    tokens = merge_compound_names(tokens)
    
    # Now, apply phonetic normalization ONLY to English-like tokens
    normalized_tokens = []
    for token in tokens:
        # A simple check: if the token consists of only ASCII letters, apply phonetic rules.
        if re.match(r'^[a-z]+$', token):
            normalized_tokens.append(phonetic_normalizer.normalize_token(token))
        else:
            # Otherwise (e.g., it's an Arabic token), keep it as is.
            normalized_tokens.append(token)
    tokens = normalized_tokens

    # 5. Final Assembly
    tokens = [t for t in tokens if len(t) >= min_token_length]
    if not tokens:
        return ""

    if remove_duplicates:
        seen = set()
        tokens = [t for t in tokens if not (t in seen or seen.add(t))]

    if sort_tokens and not preserve_order:
        tokens.sort()

    return " ".join(tokens)

# =====================================================================
# EXAMPLE USAGE
# =====================================================================

if __name__ == "__main__":
    names_to_normalize = [
        # --- Demonstrating the new phonetic normalization ---
        "Dr. Mohammed KHALED",
        "Mr. Mohamad Haled",
        "Eng. Muhammed khalid",
        "Ghaleb Charbel",
        "Galeb Sharbel",
        "Mahmoud Youssef",
        "Mahmud Yousef",
        "Tariq Shadi",
        "Tarek Sadi",
        
        # --- Demonstrating robustness with mixed and Arabic names ---
        "شركة/ أبناء يوسف وأولاده المتحدة",
        "أ.د/ مُحَمَّدٌ حُسَيْن الخالدي",
        "Mr. Abd-Elrahman Mahmoud",
        "عبد الرحمن محمود",
        "Acme Corporation Ltd. -- 1985",
        "NULL",
    ]

    print("--- Dynamic Name Normalization Pipeline ---")
    for name in names_to_normalize:
        processed_name = preprocess_name(name)
        print(f"Original:  '{name}'")
        print(f"Processed: '{processed_name}'\n")

    # Example showing how different spellings converge to one form:
    print("-" * 40)
    print("All the following variations:")
    variations = ["Mohammed", "Mohammad", "Mohamad", "Muhammed", "Mohamed","Mhmd"]
    for v in variations:
        print(f"- '{v}'")
    processed_example = preprocess_name(variations[0])
    print(f"\n...normalize to the same key: '{processed_example}'")
    print("-" * 40)

--- Dynamic Name Normalization Pipeline ---
Original:  'Dr. Mohammed KHALED'
Processed: 'haled mohamed'

Original:  'Mr. Mohamad Haled'
Processed: 'haled mohamad'

Original:  'Eng. Muhammed khalid'
Processed: 'halid muhamed'

Original:  'Ghaleb Charbel'
Processed: 'galeb sarbel'

Original:  'Galeb Sharbel'
Processed: 'galeb sarbel'

Original:  'Mahmoud Youssef'
Processed: 'iusef mahmud'

Original:  'Mahmud Yousef'
Processed: 'iusef mahmud'

Original:  'Tariq Shadi'
Processed: 'sadi tarik'

Original:  'Tarek Sadi'
Processed: 'sadi tarek'

Original:  'شركة/ أبناء يوسف وأولاده المتحدة'
Processed: 'ابناء المتحده شركه/ يوسف'

Original:  'أ.د/ مُحَمَّدٌ حُسَيْن الخالدي'
Processed: 'ا.د/ الخالدي حسين محمد'

Original:  'Mr. Abd-Elrahman Mahmoud'
Processed: 'abd-elrahman mahmud'

Original:  'عبد الرحمن محمود'
Processed: 'عبدالرحمن محمود'

Original:  'Acme Corporation Ltd. -- 1985'
Processed: '-- akme ltd.'

Original:  'NULL'
Processed: ''

----------------------------------------
All the follow

: 